# Capitolo 4 — Regressione su dati tabellari: il fabbisogno energetico di un edificio (§ 4.12)

In [1]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
fissa_seme(42)
e = dati.energia(); print(e.shape); e.describe().T[["mean", "min", "max"]]

scarico energy_efficiency.zip ... 

0.1 MB
(768, 10)


,mean,min,max
compattezza,0.764167,0.62,0.98
superficie,671.708333,514.50,808.50
sup_pareti,318.500000,245.00,416.50
sup_tetto,176.604167,110.25,220.50
altezza,5.250000,3.50,7.00
orientamento,3.500000,2.00,5.00
sup_vetrata,0.234375,0.00,0.40
distrib_vetrata,2.812500,0.00,5.00
carico_riscaldamento,22.307195,6.01,43.10
carico_raffrescamento,24.587760,10.90,48.03


In [2]:
nomi = ["compattezza", "superficie", "sup_pareti", "sup_tetto", "altezza", "orientamento", "sup_vetrata", "distrib_vetrata"]
X = e[nomi].values.astype(np.float32); y = e["carico_riscaldamento"].values.astype(np.float32)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=42)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)
scaler = StandardScaler().fit(X_tr); A, B, C = [scaler.transform(a).astype(np.float32) for a in (X_tr, X_va, X_te)]
media_y, dev_y = y_tr.mean(), y_tr.std()
mae = lambda a, b: float(np.mean(np.abs(a - b)))

In [3]:
def addestra_regressione(modello, Xtr, ytr, Xva, yva, epoche=2000, lr=1e-3, pazienza=100):
    perdita_fn = nn.MSELoss(); opt = torch.optim.Adam(modello.parameters(), lr=lr)
    Xtr, ytr, Xva, yva = map(torch.from_numpy, (Xtr, (ytr - media_y) / dev_y, Xva, (yva - media_y) / dev_y))
    migliore = (float("inf"), None, 0); attesa = 0
    for epoca in range(epoche):
        modello.train(); opt.zero_grad(); perdita_fn(modello(Xtr).squeeze(1), ytr).backward(); opt.step()
        modello.eval()
        with torch.no_grad(): lv = perdita_fn(modello(Xva).squeeze(1), yva).item()
        if lv < migliore[0]: migliore = (lv, {k: v.clone() for k, v in modello.state_dict().items()}, epoca + 1); attesa = 0
        else:
            attesa += 1
            if attesa >= pazienza: break
    modello.load_state_dict(migliore[1]); return migliore[2]

def predici(modello, X):
    modello.eval()
    with torch.no_grad(): return modello(torch.from_numpy(X)).squeeze(1).numpy() * dev_y + media_y

fissa_seme(42); lineare = nn.Linear(8, 1); ep_l = addestra_regressione(lineare, A, y_tr, B, y_va)
fissa_seme(42); mlp = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1)); ep_m = addestra_regressione(mlp, A, y_tr, B, y_va)
print(f"Linea di base (media): MAE {mae(media_y, y_te):.2f} kWh/m²")
print(f"Lineare:               MAE {mae(predici(lineare, C), y_te):.2f}   (epoca migliore {ep_l})")
print(f"MLP 8→64→64→1:        MAE {mae(predici(mlp, C), y_te):.2f}   (epoca migliore {ep_m})")

Linea di base (media): MAE 9.02 kWh/m²
Lineare:               MAE 2.43   (epoca migliore 2000)
MLP 8→64→64→1:        MAE 0.47   (epoca migliore 1993)


## Importanza per permutazione

In [4]:
base = mae(predici(mlp, C), y_te); importanza = {}
for j, nome in enumerate(nomi):
    peggioramenti = []
    for r in range(10):
        Cp = C.copy(); Cp[:, j] = np.random.default_rng(r).permutation(Cp[:, j])
        peggioramenti.append(mae(predici(mlp, Cp), y_te) - base)
    importanza[nome] = (np.mean(peggioramenti), np.std(peggioramenti))
for nome, (m, s) in sorted(importanza.items(), key=lambda t: -t[1][0]): print(f"{nome:18s} +{m:.2f} ± {s:.2f}")

altezza            +6.05 ± 0.57
sup_pareti         +3.68 ± 0.34
compattezza        +3.20 ± 0.32
sup_vetrata        +2.21 ± 0.20
sup_tetto          +2.10 ± 0.16
superficie         +1.26 ± 0.13
distrib_vetrata    +0.20 ± 0.03
orientamento       +-0.03 ± 0.03
